# Umgang mit kategorialen Variablen

## Lernziele

Am Ende dieses Notebooks sollten Sie in der Lage sein...
* herauszufinden, welche Variablen kategoriale Variablen sind.
* eine Variable mit Label-Encoder und Dummy-Variablen zu transformieren.
* Multikollinearität zu erklären und sie beim Verwenden von Dummy-Variablen zu vermeiden.

Um all dies herauszufinden, benötigen wir einen Beispieldatensatz. Bleiben wir bei den Autodaten, die wir für die multiple lineare Regression verwendet haben.

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Import data
data = pd.read_csv('data/cars_multivariate.csv')
data.head()

In [ ]:
# Check data type for each column
data.info()

Abgesehen von `car name` scheinen alle anderen Spalten numerische Werte zu haben. Sie könnten auch Kandidaten sein, um die abhängige Variable `mpg` (miles per gallon) zu beschreiben. Auf den ersten Blick scheint es, dass wir keine kategorialen Variablen haben.

## Was sind kategoriale Variablen?

Normalerweise kennen wir die Natur der Daten und wissen daher, welche Variablen kategorial oder numerisch sind. Aber selbst wenn wir die Daten nicht kennen, können wir herausfinden, ob die Variablen kategorial oder numerisch sein könnten. Schauen wir uns nun die Spalte `origin` genauer an.

In [ ]:
# Descriptive statistics for column origin
print(data['origin'].describe())

In [ ]:
# Number of unique values in column origin
print(data['origin'].nunique())

In [ ]:
# Unique values in column origin
print(data['origin'].unique())

Die Werte reichen von 1 bis 3, außerdem sind die einzigen Werte, die tatsächlich im Datensatz vorhanden sind, 1, 2 und 3! Es stellt sich heraus, dass "origin" eine sogenannte **kategoriale** Variable ist. Sie stellt keine kontinuierliche Zahl dar, sondern bezieht sich auf einen Ort - sagen wir, 1 könnte für USA stehen, 2 für Europa, 3 für Asien (Hinweis: für diesen Datensatz wird die tatsächliche Bedeutung nicht offengelegt).

Kategoriale Variablen sind dann genau das, wonach sie klingen: Sie repräsentieren Kategorien, keine numerischen Merkmale. Beachten Sie, dass diese Merkmale, obwohl hier nicht der Fall, oft als Textwerte gespeichert werden, die verschiedene Beobachtungsebenen darstellen.

## Identifizierung kategorialer Variablen

Da kategoriale Variablen auf besondere Weise behandelt werden müssen, wie Sie später sehen werden, müssen Sie sicherstellen, dass Sie identifizieren, welche Variablen kategorial sind.

In einigen Fällen ist die Identifizierung einfach (z.B. wenn sie als Strings gespeichert sind), in anderen Fällen sind sie numerisch und die Tatsache, dass sie kategorial sind, ist nicht immer sofort ersichtlich.

Beachten Sie, dass dies möglicherweise nicht trivial ist. Eine erste Sache, die Sie tun können, ist die Verwendung der `.describe()` und `.info()` Methoden. `.describe()` gibt Ihnen Informationen über die Datentypen (wie Strings, Integers usw.), aber selbst dann könnten kontinuierliche Variablen als Strings importiert worden sein, daher ist es sehr wichtig, Ihre Daten wirklich anzusehen.

In [ ]:
# Create plots to identify categorical features
fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(16,3))

for xcol, ax in zip(['acceleration', 'displacement', 'horsepower', 'weight'], axes):
    data.plot(kind='scatter', x=xcol, y='mpg', ax=ax, alpha=0.4, color='b')

In [ ]:
# Create plots to identify categorical features
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12,3))

for xcol, ax in zip([ 'cylinders', 'model_year', 'origin'], axes):
    data.plot(kind='scatter', x=xcol, y='mpg', ax=ax, alpha=0.4, color='b')

Beachten Sie den strukturellen Unterschied zwischen den oberen und unteren Diagrammgruppen. Sie können erkennen, dass die Struktur sehr unterschiedlich aussieht: Anstatt eine ziemlich homogene "Wolke" zu erhalten, erzeugen kategoriale Variablen vertikale Linien für diskrete Werte. Ein anderer Diagrammtyp, der nützlich sein kann, ist das Histogramm.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
fig = plt.figure(figsize = (8,8))
ax = fig.gca()
data.hist(ax = ax);

Und die Anzahl der eindeutigen Werte:

In [ ]:
data[['cylinders', 'model_year', 'origin']].nunique()

## Transformation kategorialer Variablen

Wenn Sie kategoriale Variablen in Regressionsmodellen verwenden möchten, müssen sie transformiert werden. Es gibt zwei Ansätze dafür:
* Label-Encoding durchführen
* Dummy-Variablen / One-Hot-Encoding erstellen

### Label-Encoding

Lassen Sie uns Label-Encoding und Dummy-Erstellung mit der folgenden pandas Series mit 3 Kategorien veranschaulichen: "USA", "EU" und "ASIA".

In [ ]:
origin = ['USA', 'EU', 'EU', 'ASIA','USA', 'EU', 'EU', 'ASIA', 'ASIA', 'USA']
origin_series = pd.Series(origin)

In [ ]:
origin_df = pd.DataFrame(origin,columns=['origin'])

Jetzt möchten Sie sicherstellen, dass Python diese Strings als Kategorien erkennt. Dies kann wie folgt erfolgen:

In [ ]:
cat_origin = origin_series.astype('category')
cat_origin

Beachten Sie, dass der `dtype` (d.h. Datentyp) hier `category` ist und die drei Kategorien erkannt werden.

Manchmal möchten Sie Ihre Labels als Zahlen darstellen. Dies wird als Label-Encoding bezeichnet.

Sie führen Label-Encoding so durch, dass numerische Labels immer zwischen 0 und (Anzahl_der_Kategorien)-1 liegen. Es gibt mehrere Möglichkeiten, dies zu tun, eine Möglichkeit ist die Verwendung von `.cat.codes`

In [ ]:
cat_origin.cat.codes

Eine andere Möglichkeit ist die Verwendung von scikit-learns `OrdinalEncoder`:

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
ord_make = OrdinalEncoder()

origin_encoded = ord_make.fit_transform(origin_df[['origin']])

In [ ]:
origin_encoded

Beachten Sie, dass `.cat.codes` nur für Variablen verwendet werden kann, die mit `.astype(category)` transformiert wurden, dies ist jedoch keine Anforderung für die Verwendung von `OrdinalEncoder`.

### Dummy-Variablen erstellen

Eine andere Möglichkeit, kategoriale Variablen zu transformieren, ist die Verwendung von One-Hot-Encoding oder "Dummy-Variablen". Die Idee ist, jede Kategorie in eine neue Spalte umzuwandeln und der Spalte eine 1 oder 0 zuzuweisen. Es gibt mehrere Bibliotheken, die One-Hot-Encoding unterstützen, schauen wir uns zwei an:

In [ ]:
# Create dummy variables using pandas .get_dummies()
pd.get_dummies(cat_origin)

Sehen Sie, wie der Label-Name zum Spaltennamen geworden ist! Eine andere Methode ist die Verwendung des `LabelBinarizer` in scikit-learn.

In [ ]:
from sklearn.preprocessing import LabelBinarizer

lb = LabelBinarizer()
origin_dummies = lb.fit_transform(cat_origin)

# You need to convert this back to a dataframe
origin_dum_df = pd.DataFrame(origin_dummies,columns=lb.classes_)
origin_dum_df

Der Vorteil der Verwendung von Dummies besteht darin, dass, unabhängig davon, welchen Algorithmus Sie verwenden werden, Ihre numerischen Werte nicht fälschlicherweise als kontinuierlich interpretiert werden können. Es ist wichtig zu wissen, dass für lineare Regression (und die meisten anderen Algorithmen in scikit-learn) **One-Hot-Encoding erforderlich ist**, wenn kategoriale Variablen in einem Regressionsmodell hinzugefügt werden!

## Die Dummy-Variablen-Falle

Aufgrund der Art und Weise, wie Dummy-Variablen erstellt werden, kann eine Variable aus allen anderen vorhergesagt werden. Dies wird als perfekte **Multikollinearität** bezeichnet und kann ein Problem für die Regression sein. Die Grundidee hinter perfekter Multikollinearität ist, dass Sie *perfekt* vorhersagen können, was eine Variable sein wird, indem Sie eine Kombination der anderen Variablen verwenden. Wenn das nicht ganz klar ist, gehen Sie zurück zu den One-Hot-codierten origin-Daten oben:

In [ ]:
# Create dummy variables using pandas .get_dummies()
trap_df = pd.get_dummies(cat_origin)
trap_df.head()

Als Folge der Erstellung von Dummy-Variablen für jedes Origin können Sie jetzt jede einzelne Origin-Dummy-Variable anhand der Informationen aus allen anderen vorhersagen. OK, das klingt vielleicht mehr wie ein Zungenbrecher als eine Erklärung, also konzentrieren Sie sich vorerst auf die ASIA-Spalte. Sie können diese Spalte perfekt vorhersagen, indem Sie die Werte in den EU- und USA-Spalten addieren und dann die Summe von 1 subtrahieren, wie unten gezeigt:

In [ ]:
# Predict ASIA column from EU and USA
predicted_asia = 1 - (trap_df['EU'] + trap_df['USA'])
predicted_asia.to_frame(name='Predicted_ASIA')

EU und USA können auf ähnliche Weise vorhergesagt werden, was Sie selbst herausarbeiten können.

Sie fragen sich wahrscheinlich, warum dies ein Problem für die Regression ist. Erinnern Sie sich daran, dass die aus einem Regressionsmodell abgeleiteten Koeffizienten zur Vorhersage verwendet werden. Bei einer multiplen linearen Regression stellen die Koeffizienten die durchschnittliche Änderung der abhängigen Variable für jede Einheitsänderung einer Prädiktorvariablen dar, vorausgesetzt, dass alle anderen Prädiktorvariablen konstant gehalten werden. Dies ist nicht mehr der Fall, wenn Prädiktorvariablen verwandt sind, was, wie Sie gerade gesehen haben, automatisch beim Erstellen von Dummy-Variablen geschieht. Dies ist die **Dummy-Variablen-Falle**.

Glücklicherweise kann die Dummy-Variablen-Falle vermieden werden, indem einfach eine der Dummy-Variablen entfernt wird. Sie können dies tun, indem Sie den Dataframe manuell unterteilen oder, bequemer, indem Sie ```drop_first=True``` an ```get_dummies()``` übergeben:

In [ ]:
pd.get_dummies(cat_origin, drop_first=True)

Wenn Sie sich den DataFrame oben genau ansehen, werden Sie sehen, dass es nicht mehr genug Informationen gibt, um eine der Spalten vorherzusagen, sodass die Multikollinearität beseitigt wurde.

Sie werden bald sehen, dass das Weglassen der ersten Variable die Interpretation der Regressionskoeffizienten beeinflusst. Die weggelassene Kategorie wird zur sogenannten **Referenzkategorie**. Die Regressionskoeffizienten, die sich aus der Anpassung der verbleibenden Variablen ergeben, stellen die Änderung *relativ* zur Referenz dar.

Sie werden auch sehen, dass in bestimmten Kontexten Multikollinearität und die Dummy-Variablen-Falle weniger problematisch sind und ignoriert werden können. Es ist daher wichtig zu verstehen, welche Modelle empfindlich auf Multikollinearität reagieren und welche nicht.

## Zurück zu unseren Autodaten

Lassen Sie uns nun unsere Spalten `cylinders`, `model_year` und `origin` in Dummies umwandeln und die erste Variable weglassen.

In [ ]:
# Create dummy variables for categorical columns
cyl_dummies = pd.get_dummies(data['cylinders'], prefix='cyl', drop_first=True)
yr_dummies = pd.get_dummies(data['model_year'], prefix='yr', drop_first=True)
orig_dummies = pd.get_dummies(data['origin'], prefix='orig', drop_first=True)

Es könnte sein, dass die Reihenfolge der Werte in den Spalten `cylinders`, `model_year` und `origin` eine Rolle spielt. Überprüfen Sie dies gerne mit dem adjustierten $R^2$, unsere Ergebnisse deuteten darauf hin, dass die Reihenfolge keine Rolle spielt. Als nächstes entfernen wir die ursprünglichen Spalten aus unseren Daten und fügen stattdessen die Dummy-Spalten hinzu

In [ ]:
# Drop original columns
data = data.drop(['cylinders','model_year','origin'], axis=1)

In [ ]:
# Concatenate dummy variables to dataframe
data = pd.concat([data, cyl_dummies, yr_dummies, orig_dummies], axis=1)
data.head()

**Probieren Sie es selbst aus**

Können Sie eine Kombination von Variablen (auch Dummy-Variablen) finden, die Ihr letztes Modell aus Notebook 5 übertrifft?

## Zusammenfassung

Sie wissen jetzt
- dass Sie Ihre Daten genau betrachten müssen, um **kategoriale Variablen zu finden** (EDA). Sie können dies tun, indem Sie sich die Informationen und deskriptiven Statistiken ansehen und herausfinden, wie viele eindeutige Werte jede Variable hat. Auch Streudiagramme und Histogramme können helfen, kategoriale Variablen zu finden.
- wie man eine Variable mit **Label-Encoder** und **Dummy-Variablen** transformiert. Für beide Methoden können Sie beispielsweise die [scikit-learns preprocessing tools](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features) verwenden.
- was **Multikollinearität** ist und wie man sie **beim Verwenden von Dummy-Variablen vermeidet**.
"Multikollinearität bezieht sich auf eine Situation, in der mehr als zwei erklärende Variablen in einem multiplen Regressionsmodell stark linear verwandt sind. Wir haben perfekte Multikollinearität, wenn beispielsweise wie in der obigen Gleichung die Korrelation zwischen zwei unabhängigen Variablen gleich 1 oder −1 ist." [Multikollinearität Wikipedia](https://en.wikipedia.org/wiki/Multicollinearity#:~:text=Multicollinearity%20refers%20to%20a%20situation,equal%20to%201%20or%20%E2%88%921.). Um Kollinearität beim Verwenden von Dummy-Variablen zu vermeiden, müssen Sie eine Dummy-Spalte weglassen.